[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C45_Privacy_Trustworthy_Course/02_dp_sgd/02_dp_sgd.ipynb)

# 02 · DP-SGD（用 numpy 从零实现）

目标：把 **per-sample 梯度裁剪**、**高斯噪声**、**简化隐私会计（子采样 RDP 直觉）**、**效用代价** 用 numpy 实现，并 `assert` 验证。

路线：逐样本梯度 → 裁剪(=限制敏感度) → 加噪求和 → 一步 DP-SGD → 训练 + 效用代价 → 会计(ε vs σ/q/T) → ✏️ 练习 → 📖 答案 → 🧪 真实超参胶囊。

> 心智模型：**DP-SGD 一步 = 模块 01 的高斯机制**。敏感度 = 裁剪范数 C；噪声 = N(0,σ²C²)；模型 = 带噪梯度的后处理 -> 自动私有。

## 1 · 玩具问题：逻辑回归 + 逐样本梯度

用一个可分的二分类逻辑回归当靶子（纯 numpy）。先实现**逐样本梯度**——DP-SGD 的起点（普通 SGD 只需 batch 平均，DP-SGD 必须先拿到每条样本的梯度才能逐个裁剪）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_data(n=500, d=10, seed=0):
    g = np.random.default_rng(seed)
    w_true = g.standard_normal(d)
    X = g.standard_normal((n, d))
    logits = X @ w_true
    y = (logits + 0.3*g.standard_normal(n) > 0).astype(float)
    return X, y

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def per_sample_grads(w, X, y):
    '''返回每条样本的梯度，形状 (n, d)。逻辑回归: g_i = (sigmoid(x_i·w) - y_i) * x_i。'''
    preds = sigmoid(X @ w)                 # (n,)
    return (preds - y)[:, None] * X         # (n, d)

X, y = make_data()
w = np.zeros(X.shape[1])
G = per_sample_grads(w, X, y)
print(f'逐样本梯度形状 = {G.shape}  (n 条样本各一个 d 维梯度)')
# 对拍：逐样本梯度的均值 == batch 平均梯度（普通 SGD 用的那个）
batch_grad = G.mean(axis=0)
preds = sigmoid(X @ w)
ref_grad = X.T @ (preds - y) / len(y)
assert np.allclose(batch_grad, ref_grad), '逐样本梯度均值应=batch平均梯度'
print('✅ 逐样本梯度正确：其均值 == 普通 SGD 的 batch 平均梯度')

## 2 · 逐样本裁剪：把单样本影响（敏感度）限制在 C

裁剪：`ḡ_i = g_i · min(1, C/‖g_i‖₂)`。裁剪后每条梯度 L2 范数 ≤ C，于是「增删一个样本」对**梯度和**的影响 ≤ C —— 这就是 **L2 敏感度 = C**。

**关键**：必须逐样本裁剪再求和，不能裁剪 batch 平均梯度。

In [ ]:
def clip_per_sample(G, C):
    '''逐样本 L2 裁剪到范数上界 C。G:(n,d) -> (n,d)。'''
    norms = np.linalg.norm(G, axis=1, keepdims=True)       # (n,1)
    factors = np.minimum(1.0, C / (norms + 1e-12))
    return G * factors

C = 1.0
G = per_sample_grads(rng.standard_normal(X.shape[1]), X, y)  # 用非零 w 制造大梯度
Gc = clip_per_sample(G, C)
clipped_norms = np.linalg.norm(Gc, axis=1)
print(f'裁剪前最大范数 = {np.linalg.norm(G,axis=1).max():.3f}')
print(f'裁剪后最大范数 = {clipped_norms.max():.3f}  (应 ≤ C={C})')
assert clipped_norms.max() <= C + 1e-9, '裁剪后所有范数应≤C'
# 敏感度验证：去掉任一样本，梯度和变化的 L2 ≤ C
g_sum = Gc.sum(axis=0)
max_change = max(np.linalg.norm(g_sum - (g_sum - Gc[i])) for i in range(len(Gc)))
print(f'去掉单样本对梯度和的最大变化 L2 = {max_change:.3f}  (应 ≤ C={C})')
assert max_change <= C + 1e-9, '单样本对梯度和的影响=L2敏感度，应≤C'
print('✅ 逐样本裁剪 -> 梯度和的 L2 敏感度 = C，可用高斯机制标定噪声')

## 3 · 加高斯噪声：DP-SGD 一步 = 高斯机制

对裁剪梯度的**和**加 `N(0, σ²C²)` 噪声，再平均。这一步就是模块 01 的高斯机制（敏感度=C，噪声 std=σC）。

验证：加噪后的梯度估计**无偏**（期望=裁剪梯度均值），噪声尺度正确。

In [ ]:
def private_grad(G, C, sigma, rng):
    '''DP-SGD 的私有梯度：逐样本裁剪 -> 求和 -> 加 N(0,σ²C²) -> 平均。'''
    n = len(G)
    Gc = clip_per_sample(G, C)
    noisy_sum = Gc.sum(axis=0) + rng.normal(0.0, sigma * C, size=G.shape[1])
    return noisy_sum / n

C, sigma = 1.0, 1.0
G = per_sample_grads(np.zeros(X.shape[1]), X, y)
Gc_mean = clip_per_sample(G, C).mean(axis=0)
# 跑很多次取平均，应回到裁剪梯度均值（无偏）
draws = np.array([private_grad(G, C, sigma, rng) for _ in range(3000)])
print(f'私有梯度均值 vs 裁剪梯度均值 max|diff| = {np.abs(draws.mean(axis=0)-Gc_mean).max():.4f}')
assert np.allclose(draws.mean(axis=0), Gc_mean, atol=0.02), '私有梯度应无偏(对裁剪梯度均值)'
# 每维噪声 std = σC/n
exp_std = sigma * C / len(G)
assert np.allclose(draws.std(axis=0), exp_std, atol=exp_std*0.2), '噪声尺度应=σC/n'
print(f'✅ DP-SGD 一步 = 高斯机制：无偏 + 每维噪声 std≈σC/n={exp_std:.5f}')

## 4 · 完整 DP-SGD 训练 vs 非私有 SGD：看效用代价

把上面拼成完整训练循环，和普通（非私有）SGD 对拍**准确率差距** —— 这就是隐私的效用代价。

In [ ]:
def train(X, y, steps=300, lr=0.5, C=1.0, sigma=0.0, batch=64, rng=None):
    '''sigma=0 即普通 SGD（无噪声、无裁剪影响可忽略）；sigma>0 即 DP-SGD。'''
    rng = rng or np.random.default_rng(0)
    n, d = X.shape
    w = np.zeros(d)
    for t in range(steps):
        idx = rng.choice(n, size=batch, replace=False)    # 子采样一个 batch
        Xb, yb = X[idx], y[idx]
        G = per_sample_grads(w, Xb, yb)
        if sigma > 0:
            grad = private_grad(G, C, sigma, rng)         # DP-SGD
        else:
            grad = G.mean(axis=0)                          # 普通 SGD
        w -= lr * grad
    return w

def accuracy(w, X, y):
    return float(np.mean((sigmoid(X @ w) > 0.5) == (y > 0.5)))

Xtr, ytr = make_data(n=2000, d=10, seed=1)
Xte, yte = make_data(n=1000, d=10, seed=2)
# 注意：测试集用同一 w_true（seed 决定 w_true），这里 seed 不同会换 w_true；改用同分布
g = np.random.default_rng(7); w_true = g.standard_normal(10)
def gen(n, seed):
    gg = np.random.default_rng(seed); X = gg.standard_normal((n,10))
    y = (X @ w_true + 0.3*gg.standard_normal(n) > 0).astype(float); return X, y
Xtr, ytr = gen(2000, 1); Xte, yte = gen(1000, 2)

w_sgd = train(Xtr, ytr, sigma=0.0, rng=np.random.default_rng(0))
w_dp  = train(Xtr, ytr, sigma=1.0, C=1.0, rng=np.random.default_rng(0))
acc_sgd = accuracy(w_sgd, Xte, yte)
acc_dp  = accuracy(w_dp,  Xte, yte)
print(f'非私有 SGD 测试准确率 = {acc_sgd:.3f}')
print(f'DP-SGD(σ=1) 测试准确率 = {acc_dp:.3f}')
print(f'效用代价 = {acc_sgd - acc_dp:.3f}')
assert acc_sgd > 0.8, '非私有 SGD 应学得不错'
assert acc_dp > 0.65, 'DP-SGD 仍应学到有用信号(高于随机0.5)'
assert acc_dp <= acc_sgd + 0.02, 'DP-SGD 准确率不应超过非私有(噪声只会伤害)'
print('✅ DP-SGD 学到了有用信号，但有可量化的效用代价 —— 没有免费的隐私')

**噪声越大 -> 效用越差**。扫不同 σ，看准确率如何随隐私强度下降。

In [ ]:
print(f"{'σ(噪声乘子)':>12s} {'测试准确率':>12s}  {'隐私':<6s}")
accs = []
for sigma in [0.0, 0.5, 1.0, 2.0, 4.0]:
    w = train(Xtr, ytr, sigma=sigma, C=1.0, rng=np.random.default_rng(0))
    a = accuracy(w, Xte, yte)
    accs.append(a)
    tag = '无' if sigma==0 else ('弱' if sigma<=1 else '强')
    print(f'{sigma:>12.1f} {a:>12.3f}  {tag:<6s}')
# 大体单调：噪声越大准确率越低（允许小波动，故比较两端）
assert accs[0] >= accs[-1] - 0.02, 'σ 越大准确率大体越低'
assert accs[-1] > 0.5, '即便强噪声也应略好于随机'
print('\n✅ 噪声乘子 σ 越大 -> 隐私越强 -> 准确率越低。DP-SGD 的效用代价。')

## 5 · 隐私会计：ε 随 σ / q / T 怎么变

为什么 T 步没把 ε 炸掉？靠**子采样放大** + **RDP 紧致组合**。我们用一个**简化但趋势正确**的会计器（基于子采样高斯的 RDP 上界）模拟它，看 ε 随三个旋钮的变化。

> 这是教学用的简化会计（趋势对、数量级对），真实 Opacus 的 RDP accountant 更精确。重点是建立 **ε↓ when σ↑、ε↑ when T↑、ε↓ when q↓** 的直觉。

In [ ]:
def rdp_subsampled_gaussian(q, sigma, steps, orders=None):
    '''简化 RDP 会计：子采样高斯每步的 RDP(α) ≈ q²α/σ²（小 q 近似），T 步线性累加，
       再在多个阶 α 上转换回 (ε,δ) 取最优。趋势正确、用于教学。'''
    if orders is None:
        orders = np.array([1.5, 2, 4, 8, 16, 32, 64])
    delta = 1e-5
    best_eps = np.inf
    for a in orders:
        rdp_step = q*q * a / (sigma*sigma)          # 子采样高斯单步 RDP(α) 的小q近似
        rdp_total = rdp_step * steps                 # RDP 直接相加（紧致组合）
        eps = rdp_total + np.log(1/delta) / (a - 1)  # RDP -> (ε,δ) 标准转换
        best_eps = min(best_eps, eps)
    return float(best_eps)

# 基准：q=0.01, σ=1.0, T=1000
base = rdp_subsampled_gaussian(0.01, 1.0, 1000)
print(f'基准 (q=0.01, σ=1.0, T=1000):  ε ≈ {base:.3f}')
print(f'  σ 翻倍(1->2):   ε ≈ {rdp_subsampled_gaussian(0.01, 2.0, 1000):.3f}  (应更小)')
print(f'  T 翻倍(1k->2k): ε ≈ {rdp_subsampled_gaussian(0.01, 1.0, 2000):.3f}  (应更大)')
print(f'  q 翻倍(.01->.02):ε ≈ {rdp_subsampled_gaussian(0.02, 1.0, 1000):.3f}  (应更大)')
assert rdp_subsampled_gaussian(0.01,2.0,1000) < base, 'σ 越大 ε 越小(更私密)'
assert rdp_subsampled_gaussian(0.01,1.0,2000) > base, 'T 越大 ε 越大(损失累加)'
assert rdp_subsampled_gaussian(0.02,1.0,1000) > base, 'q 越大 ε 越大(每步碰到的人多)'
print('✅ ε: σ↑→小、T↑→大、q↑→大。这就是 DP-SGD 三旋钮的隐私账。')

**ε 随 T 是 √T 而非 T**（紧致组合的威力）。对拍：朴素线性组合 vs RDP 会计，看后者紧多少。

In [ ]:
q, sigma = 0.01, 1.0
print(f"{'T':>8s} {'朴素线性∝T':>14s} {'RDP会计':>12s} {'紧致比':>8s}")
rdp_1k = rdp_subsampled_gaussian(q, sigma, 1000)
for T in [1000, 4000, 16000]:
    naive = rdp_1k * (T/1000)                 # 假想线性增长(∝T)
    rdp = rdp_subsampled_gaussian(q, sigma, T)
    print(f'{T:>8d} {naive:>14.3f} {rdp:>12.3f} {naive/rdp:>7.1f}x')
# T 增大 16 倍，RDP 的 ε 增长应远小于 16 倍（次线性）
ratio = rdp_subsampled_gaussian(q,sigma,16000) / rdp_1k
print(f'\nT 增大 16x，RDP 的 ε 只增大 {ratio:.1f}x (远小于 16x -> 次线性)')
assert ratio < 16, 'RDP 会计下 ε 增长应次线性于 T'
print('✅ 紧致会计让 ε 增长远慢于线性 —— 这就是 DP-SGD 能训很多步还可用的原因')

---
## ✏️ 练习 1：实现逐样本梯度裁剪

实现 `clip_to_norm(G, C)`：把每行（每条样本梯度）的 L2 范数裁剪到 ≤ C，方向不变。

In [ ]:
def clip_to_norm(G, C):
    # TODO: 对 G(n,d) 每行算 L2 范数，若 > C 则缩放到范数=C；返回 (n,d)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
G = np.random.default_rng(3).standard_normal((50, 8)) * 5   # 大范数
Gc = clip_to_norm(G, 1.0)
norms = np.linalg.norm(Gc, axis=1)
assert norms.max() <= 1.0 + 1e-9, '所有范数应≤C'
# 方向不变：裁剪后梯度与原梯度同向
i = np.argmax(np.linalg.norm(G, axis=1))
cos = G[i] @ Gc[i] / (np.linalg.norm(G[i])*np.linalg.norm(Gc[i]) + 1e-12)
assert cos > 0.999, '裁剪应只改幅度不改方向'
# 小范数的梯度不应被改动
small = np.random.default_rng(4).standard_normal((5,8)) * 0.01
assert np.allclose(clip_to_norm(small, 1.0), small), '范数<C 的梯度不应被裁剪'
print('✅ 练习 1 通过：逐样本裁剪范数正确、方向不变、小梯度不动')

## ✏️ 练习 2：高斯噪声标定

DP-SGD 一步要对裁剪梯度和加 `N(0, (σC)²)` 噪声。实现 `add_dp_noise(grad_sum, C, sigma, rng)`：给梯度和加正确标定的高斯噪声。

In [ ]:
def add_dp_noise(grad_sum, C, sigma, rng):
    # TODO: 给 grad_sum(d,) 加 N(0, (σ·C)²) 的高斯噪声并返回
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rng_t = np.random.default_rng(5)
gs = np.array([3.0, -2.0, 1.0])
C, sigma = 2.0, 1.5
draws = np.array([add_dp_noise(gs, C, sigma, rng_t) for _ in range(20000)])
assert np.allclose(draws.mean(axis=0), gs, atol=0.1), '加噪应无偏'
exp_std = sigma * C
assert np.allclose(draws.std(axis=0), exp_std, atol=exp_std*0.1), f'噪声 std 应=σC={exp_std}'
print(f'噪声 std ≈ {draws.std(axis=0).mean():.3f} (期望 σC={exp_std})')
print('✅ 练习 2 通过：高斯噪声按 σC 标定正确')

## ✏️ 练习 3：隐私会计（ε 随 σ 单调）

用给定的简化会计器 `rdp_subsampled_gaussian(q, sigma, steps)`，实现 `sigma_for_target_eps(q, steps, target_eps)`：用二分搜索找到达到目标 ε 所需的**最小**噪声乘子 σ（ε 随 σ 单调下降，所以可二分）。

In [ ]:
def sigma_for_target_eps(q, steps, target_eps, lo=0.1, hi=100.0, iters=60):
    # TODO: 二分搜索 σ，使 rdp_subsampled_gaussian(q,sigma,steps) ≈ target_eps
    #       ε 随 σ 单调下降：σ 太小 -> ε 太大。返回满足 ε<=target 的最小 σ
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
q, steps, target = 0.01, 1000, 1.0
sigma = sigma_for_target_eps(q, steps, target)
got_eps = rdp_subsampled_gaussian(q, sigma, steps)
print(f'目标 ε={target} -> 需要 σ≈{sigma:.3f} (实得 ε={got_eps:.3f})')
assert abs(got_eps - target) < 0.05, '找到的 σ 应使 ε≈目标'
# 更严的隐私(更小 target_eps)需要更大的 σ
sigma_strict = sigma_for_target_eps(q, steps, 0.5)
assert sigma_strict > sigma, '更小的目标 ε 需要更大的 σ'
print('✅ 练习 3 通过：能反解出达到目标 ε 的噪声乘子（隐私会计的核心用法）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def clip_to_norm(G, C):
    norms = np.linalg.norm(G, axis=1, keepdims=True)
    return G * np.minimum(1.0, C / (norms + 1e-12))

In [ ]:
# 练习 2 参考答案
def add_dp_noise(grad_sum, C, sigma, rng):
    return grad_sum + rng.normal(0.0, sigma * C, size=np.shape(grad_sum))

In [ ]:
# 练习 3 参考答案
def sigma_for_target_eps(q, steps, target_eps, lo=0.1, hi=100.0, iters=60):
    for _ in range(iters):
        mid = 0.5*(lo+hi)
        if rdp_subsampled_gaussian(q, mid, steps) > target_eps:
            lo = mid          # ε 太大 -> 需要更大 σ
        else:
            hi = mid
    return hi

---
## 🧪 真实数据胶囊：复刻 Abadi 2016 的隐私-效用权衡表

Abadi 2016（DP-SGD 原论文）在 MNIST 上报告了不同 ε 对应的准确率。我们用论文量级的**真实数字**复刻这条权衡曲线，并用我们的会计器反推：要达到这些 ε，需要多大的 σ。

（数字取自论文与后续工作的公开报告，量级真实；本环境不联网、不训 MNIST，用内置数值。）

In [ ]:
# Abadi 2016 / 后续工作在 MNIST 上的代表性 (ε, 准确率) —— 量级真实
MNIST_DP = [
    # (ε, 约准确率%, 备注)
    (0.5,  90.0, '强隐私'),
    (2.0,  95.0, '中等'),
    (8.0,  97.0, '弱隐私'),
    (np.inf, 99.0, '非私有(对照)'),
]
print(f"{'ε':>8s} {'准确率%':>8s}  {'备注':<14s}")
accs = []
for eps, acc, note in MNIST_DP:
    print(f'{eps:>8.1f} {acc:>8.1f}  {note:<14s}')
    accs.append(acc)
# 单调性：ε 越大(越不私密) -> 准确率越高
assert accs == sorted(accs), 'ε 越大准确率应越高(隐私-效用权衡)'
# 用会计器反推 ε=2.0 需要的 σ（MNIST 典型 q、T）
q_mnist, T_mnist = 256/60000, 10000      # batch=256, 60k 训练样本, 1万步
sigma_needed = sigma_for_target_eps(q_mnist, T_mnist, 2.0)
print(f'\n要在 MNIST 上达到 ε=2.0 (q={q_mnist:.4f}, T={T_mnist}): 需 σ≈{sigma_needed:.3f}')
assert sigma_needed > 0, '应能反解出所需噪声乘子'
print('✅ 复刻隐私-效用权衡：强隐私(小ε)掉点明显，弱隐私接近非私有 —— 与 Abadi 2016 一致')

**🧪 胶囊练习**：实现 `utility_drop(dp_table)`：给定 (ε, 准确率) 表，返回一个字典 `{ε: 相对非私有的准确率下降}`，量化每个隐私强度的「代价」。验证：ε 越小，代价越大。

In [ ]:
def utility_drop(dp_table):
    # TODO: 找到 ε=inf 的非私有准确率作基准，返回 {ε: 基准 - 该ε准确率}（不含 inf 本身）
    raise NotImplementedError

In [ ]:
# 自测
drops = utility_drop(MNIST_DP)
print('各 ε 的效用代价(准确率下降):', {k: round(v,1) for k,v in drops.items()})
assert drops[0.5] > drops[8.0], 'ε 越小代价越大'
assert abs(drops[2.0] - 4.0) < 0.5, 'ε=2.0 代价应≈99-95=4'
print('✅ 胶囊练习通过：量化了不同隐私强度的效用代价')

In [ ]:
# 📖 胶囊参考答案
def utility_drop(dp_table):
    baseline = next(a for e, a, _ in dp_table if e == np.inf)
    return {e: baseline - a for e, a, _ in dp_table if e != np.inf}

### 小结
- **DP-SGD 一步 = 高斯机制**：逐样本裁剪(敏感度=C) -> 求和加 N(0,σ²C²) -> 平均；模型=带噪梯度后处理->自动私有。
- **必须逐样本裁剪**（`sum(clip(gᵢ))` 而非 `clip(mean(g))`），否则敏感度无界、隐私不成立。
- **T 步没炸掉 ε**：子采样放大(q↓→更私密) + RDP/moments accountant 紧致组合(ε∝√T 而非 T)。
- **效用代价**来自裁剪偏差 + 噪声方差；大数据/大batch/调好超参可大幅压低（DP 并非必然没法用）。
- **端到端隐私**：跑了 DP-SGD ≠ 整个流程私有；超参搜索、BatchNorm、数据预处理都要核算。

下一站：**模块 03 · 联邦学习** —— 换一个正交的轴：不加噪，而是让数据根本不离开设备。